# Laya V4.6 — Publish benchmark reports to Hugging Face

This helper publishes **experiment metrics only** to a Hugging Face **Dataset repository**, using the ZIP archive supplied separately by ChatGPT. No pretrained weights, tokens, or raw private task examples are included.

**Why login is required:** The connected Hugging Face OAuth session has read-only permissions. This notebook requests your own explicit write-authorized login to Hugging Face.

1. Download `laya_v46_hf_results.zip` from the accompanying ChatGPT answer.
2. Run all cells in this notebook.
3. Select the downloaded ZIP when the upload dialog appears.
4. Authorize a Hugging Face token with permission to write to repositories under your account.
5. Default is **private**; edit `PRIVATE = False` only if you want to make results public.

This notebook refuses to overwrite an existing dataset repository, and validates the uploaded archive.


In [ ]:
#@title 1. Install Hugging Face Hub client
%pip -q install -U "huggingface_hub>=0.25,<2"
from huggingface_hub import HfApi, notebook_login
from pathlib import Path
import tempfile, json, zipfile, hashlib


In [ ]:
#@title 2. Upload and validate laya_v46_hf_results.zip
from google.colab import files
uploads=files.upload()
assert len(uploads)==1,"Select exactly one laya_v46_hf_results.zip file."
filename=next(iter(uploads))
assert filename.endswith(".zip"),"Expected the ZIP created by ChatGPT"
root=Path(tempfile.mkdtemp(prefix="laya_v46_hf_"))
expected={
"README.md","manifest.json","benchmark_summary.csv",
"metrics/v46_runtime_probe.json","metrics/v46_probe_sweep.json",
"metrics/v46_final_report.json","metrics/v46_distill_probe.json"}
with zipfile.ZipFile(filename) as archive:
    members=set(archive.namelist())
    assert members==expected,f"Unexpected or missing archive contents: {members.symmetric_difference(expected)}"
    for name in members:
        out=root/name
        out.parent.mkdir(parents=True,exist_ok=True)
        out.write_bytes(archive.read(name))
manifest=json.loads((root/"manifest.json").read_text())
for name,expected_hash in manifest["sha256"].items():
    digest=hashlib.sha256((root/name).read_bytes()).hexdigest()
    assert digest==expected_hash,f"Checksum failed: {name}"
report=json.loads((root/"metrics/v46_final_report.json").read_text())
assert report["version"]=="V4.6" and report["selected_context_budget"]==64
assert report["gates"]["all_pass"] is True
print("Validated",len(expected),"files from",filename)
print("Selected configuration:",report["selected_runtime"],"budget",report["selected_context_budget"])


In [ ]:
#@title 3. Sign into your Hugging Face account with WRITE permissions
# Hugging Face never stores your access token in this notebook or the dataset.
notebook_login()
api=HfApi()
identity=api.whoami()
owner=identity["name"]
print("Signed in as:",owner)


In [ ]:
#@title 4. Create or resume an EMPTY private dataset repo and upload reports
REPO_NAME="laya-v46-fidelity-speed-audit-20261013" #@param {type:"string"}
PRIVATE=True #@param {type:"boolean"}
assert REPO_NAME and "/" not in REPO_NAME
repo_id=f"{owner}/{REPO_NAME}"
url=f"https://huggingface.co/datasets/{repo_id}"
if api.repo_exists(repo_id,repo_type="dataset"):
    existing=set(api.list_repo_files(repo_id=repo_id,repo_type="dataset"))
    # A previous interrupted run may have created the empty repository.
    # Only allow retry if empty or containing exclusively the same report filenames.
    if existing - expected - {".gitattributes",".gitignore"}:
        raise RuntimeError(f"{repo_id} contains unrelated files: {existing-expected}. Refusing to overwrite.")
    print("Resuming existing dataset repository:",url)
    if expected.issubset(existing):
        print("Results already present: verifying uploaded files (no overwrite).")
        for path in sorted(expected):
            from huggingface_hub import hf_hub_download
            remote=hf_hub_download(repo_id=repo_id,filename=path,repo_type="dataset")
            assert Path(remote).read_bytes()==(root/path).read_bytes(),f"Remote mismatch: {path}"
        print("Already uploaded and verified:",url)
    else:
        api.upload_folder(repo_id=repo_id,repo_type="dataset",folder_path=str(root),
                          commit_message="Publish V4.6 marker-aware decision-head metrics")
else:
    created=api.create_repo(repo_id=repo_id,repo_type="dataset",
                             private=PRIVATE,exist_ok=False)
    print("Created dataset:",str(created))
    api.upload_folder(repo_id=repo_id,repo_type="dataset",folder_path=str(root),
                      commit_message="Publish V4.6 marker-aware decision-head metrics")
listed=set(api.list_repo_files(repo_id=repo_id,repo_type="dataset"))
assert expected.issubset(listed),f"Missing remote files: {expected-listed}"
print("Upload VERIFIED:",url)
print("Private repo:" if PRIVATE else "Public repo:",repo_id)
